### `Step-2` : Image Resolution Filter
- In the model training process, having high-quality images improves the model's learning accuracy. After collecting around 3,000 images each animals, the dataset was transferred to a VM system for faster processing using Apache PySpark. A shared file system between VM and Windows was used to facilitate seamless data transfer and processing.
- After that, the dataset was transferred from the local system to the HDFS (Hadoop Distributed File System) for storage and processing. The following code lines were executed in the Ubuntu terminal to perform the transfer:
  - hdfs dfs -mkdir -p /user/dangerous_animalsn  - hdfs dfs -put /home/student/Desktop/dangerous_animals /user/dangerous_animals/datasets
  - hdfs dfs -chmod 777 /user/dangerous_animals
  - hdfs dfs -ls /user/dangerous_animals
- After transferring the data to the HDFS system, I applied an image resolution filter to remove low-resolution images across all image types.


In [ ]:
from pyspark.sql import SparkSession
from PIL import Image
from io import BytesIO

# Initialize or reuse SparkSession
if 'sc' in globals():
    spark = SparkSession.builder.getOrCreate()
else:
    spark = SparkSession.builder.appName("ImageFilterLowPixel").getOrCreate()

# HDFS path to the target folder
hdfs_folder = "hdfs://localhost:9000/user/dangerous_animals/datasets"

# Pixel threshold for deletion (width, height)
pixel_threshold = (1000, 1000)

# Function to check if an image is below the pixel threshold
def is_low_pixel_image(file_path):
    try:
        # Read image from HDFS
        image_data = spark.sparkContext.binaryFiles(file_path).collect()[0][1]
        image = Image.open(BytesIO(image_data))
        width, height = image.size
        
        # Return True if dimensions are below threshold
        return width < pixel_threshold[0] or height < pixel_threshold[1]
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return False

# List files in the HDFS folder
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
fs = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
status = fs.listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path(hdfs_folder))

# Filter image files
image_files = [file.getPath().toString() for file in status if file.getPath().toString().endswith(('.jpg', '.jpeg', '.png'))]

# Debug: Print found files
print(f"Files found in {hdfs_folder}:")
for file in image_files:
    print(file)

# Track deleted files
deleted_count = 0

# Delete low-resolution images
for file_path in image_files:
    print(f"Checking file: {file_path}")
    if is_low_pixel_image(file_path):
        print(f"Deleting: {file_path}")
        try:
            fs.delete(spark.sparkContext._jvm.org.apache.hadoop.fs.Path(file_path), True)
            print(f"Deleted: {file_path}")
            deleted_count += 1
        except Exception as e:
            print(f"Failed to delete {file_path}: {e}")

# Summary
print(f"Total deleted images: {deleted_count}")
print("Processing completed.")
